# Agent: forecast

Develop and test **`agentic_scd.agents.forecast.forecast_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    UP["classifications<br/>Classification[]"]:::faded --> C1
    subgraph C["forecast_node (batch aggregate)"]
        C1["aggregate_risk = mean(risk_score)"] --> C2["baseline = flat demand"]
        C1 --> C3["adjusted[t] = baseline * (1 - risk * (t+1)/H)"]
    end
    C2 --> D["forecast<br/>Forecast(baseline, adjusted, note)"]
    C3 --> D
    D --> DOWN["downstream: (read by dashboard)"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `classifications` (for aggregate risk)
- **Writes:** `forecast` (`Forecast`: baseline, adjusted, note)
- **Fallback / degradation:** no classifications → aggregate risk 0.0 → adjusted equals baseline (no bend)

**Phase 5** replaces this with a Prophet baseline + risk-adjusted forecast, behind the same `forecast_node` signature.

## Is the DB up? (optional)

In [ ]:
# Optional: this agent runs fine offline on synthetic sample state. This snippet just
# reports whether the live DB is reachable (Setup section of 00_orchestration brings
# it up).
from agentic_scd.devtools import db_status

status = db_status()
print(status.detail)
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")

## Build a representative input state

In [ ]:
from agentic_scd.agents.classify import classify_node
from agentic_scd.devtools import sample_state

state = sample_state(count=2)
state.update(classify_node(state))  # forecast reads classifications

## Call `forecast_node` in isolation

In [ ]:
from agentic_scd.agents.forecast import aggregate_risk, forecast_node

print("aggregate risk:", round(aggregate_risk(state["classifications"]), 3))
state.update(forecast_node(state))
f = state["forecast"]
print("baseline:", f.baseline)
print("adjusted:", f.adjusted)
print(f.note)

## Iterate here

This is your dev surface: tweak the input above, re-run, and watch `forecast_node`'s output change. When you deepen this agent in its phase, keep the node signature the same so the rest of the graph is unaffected.